# Chapter 10: Basic Regression Analysis with Time Series Data

Topics: static time series models, equispaced and irregular time series, FDL models, trends, seasonality.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 10.1 Static Time Series Models

## Step 2 – Static Time Series Models

A **static model** relates the contemporaneous values of $y$ and $z$:
$$y_t = \beta_0 + \beta_1 z_t + u_t$$

Key assumptions for unbiasedness: **strict exogeneity** — $E(u_t|z_1,\ldots,z_n)=0$.

⚠️ Serial correlation and heteroscedasticity are common in time series data — always inspect residuals and use robust SEs when appropriate.

In [ ]:
df = woo.dataWoo('phillips')
df.index = pd.date_range(start='1948', periods=len(df), freq='YE')

print(df.head())
print(f"\nSample: {df.index[0].year} – {df.index[-1].year}")

model = smf.ols('inf ~ unem', data=df).fit()
print(model.summary())

## 10.2 Time Series Plots

## Step 3 – Finite Distributed Lag (FDL) Models

An **FDL(q)** model allows $z$ to affect $y$ with a delay:
$$y_t = \alpha + \delta_0 z_t + \delta_1 z_{t-1} + \cdots + \delta_q z_{t-q} + u_t$$

**Impact multiplier:** $\delta_0$ — immediate effect of a unit increase in $z$.

**Cumulative multiplier after $h$ periods:** $\sum_{j=0}^h \delta_j$

**Long-run multiplier (LRM):** total effect of a permanent unit increase in $z$:
$$LRM = \sum_{j=0}^q \delta_j$$

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df.index, df['inf'],  color='steelblue', lw=2)
axes[0].set_ylabel('Inflation Rate (%)'); axes[0].set_title('US Inflation')
axes[0].axhline(df['inf'].mean(), color='red', linestyle='--', alpha=0.7, label='Mean')
axes[0].legend()

axes[1].plot(df.index, df['unem'], color='darkorange', lw=2)
axes[1].set_ylabel('Unemployment Rate (%)'); axes[1].set_title('US Unemployment')

plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_time_series_plots.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_10_Basic_Time_Series_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10.3 Finite Distributed Lag Models

## Step 4 – Trends and Seasonality

**Linear time trend:**
$$y_t = \beta_0 + \beta_1 t + u_t$$

**Exponential trend** (use log form):
$$\log(y_t) = \beta_0 + \beta_1 t + u_t$$

where $100\hat{\beta}_1$ is the approximate annual percentage growth rate.

Including a trend in regression **removes spurious correlation** between trending variables. This is equivalent to detrending both $y$ and $x$ before regressing.

In [ ]:
df['unem_lag1'] = df['unem'].shift(1)
df['unem_lag2'] = df['unem'].shift(2)

m_fdl = smf.ols('inf ~ unem + unem_lag1 + unem_lag2', data=df.dropna()).fit()
print(m_fdl.summary())

# Long-run multiplier
lrm = m_fdl.params['unem'] + m_fdl.params['unem_lag1'] + m_fdl.params['unem_lag2']
print(f"\nLong-run multiplier: {lrm:.4f}")

## 10.3.2 Trends

In [ ]:
df['t'] = np.arange(1, len(df) + 1)
m_trend = smf.ols('inf ~ unem + t', data=df.dropna()).fit()
print("Model with linear trend:")
print(m_trend.summary())